# 01: Translation run

The **only** notebook that calls the LLM. For each cell in `RUN_MATRIX` it drives `SQLTranslator` over every gold query and writes one record per query to `records_<dataset>_<target>_<model>.json` in `evaluation/outputs/`. Downstream notebooks glob those files.

Configuration lives in `eval_harness.config` (temperature 0.0, max 3 iterations, syntax validation for Cypher). Token counts come straight from `result.token_usage` -- no log scraping. Resume is automatic: query ids already on disk for a cell are skipped.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "evaluation"))

from dataclasses import replace
from eval_harness import RUN_MATRIX, run_translation, load_records, OUTPUTS_DIR

for rc in RUN_MATRIX:
    print(rc)

RunConfig(dataset='ldbc', target='cypher', model='qwen3-coder:30b', provider='ollama', validation_mode=None, max_iterations=3, temperature=0.0, num_ctx=16384, host='http://localhost:11434', outputs_dir=PosixPath('/Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs'), subset=None, resume=True, server_config=None)
RunConfig(dataset='ldbc', target='cypher', model='claude-opus-4-8', provider='anthropic', validation_mode=None, max_iterations=3, temperature=0.0, num_ctx=16384, host='http://localhost:11434', outputs_dir=PosixPath('/Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs'), subset=None, resume=True, server_config=None)


## Optional smoke subset

Uncomment to restrict every matrix cell to one query before paying for a full run.

In [2]:
# RUN_MATRIX = [replace(rc, subset=('ldbc_q01',)) for rc in RUN_MATRIX]
print('Will run', len(RUN_MATRIX), 'matrix cell(s).')

Will run 2 matrix cell(s).


## Run

Writes incrementally; safe to interrupt and resume.

In [3]:
for rc in RUN_MATRIX:
    run_translation(rc)

Resume: 14 record(s) on disk for records_ldbc_cypher_qwen3-coder_30b.json; 14 query id(s) done.
ldbc/cypher/qwen3-coder:30b: 0 item(s) to translate.
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_cypher_qwen3-coder_30b.json
Resume: 1 record(s) on disk for records_ldbc_cypher_claude-opus-4-8.json; 1 query id(s) done.
ldbc/cypher/claude-opus-4-8: 13 item(s) to translate.
[  1/13] ldbc_q02 -> cypher 

ok iters=1 tokens=(   71,  71)   3.2s status=success
[  2/13] ldbc_q03 -> cypher 

ok iters=1 tokens=(   55,  45)   1.7s status=success
[  3/13] ldbc_q04 -> cypher 

ok iters=1 tokens=(  115,  82)   2.3s status=success
[  4/13] ldbc_q05 -> cypher 

ok iters=1 tokens=(  159,  95)   2.3s status=success
[  5/13] ldbc_q06 -> cypher 

...

ok iters=1 tokens=(  117,  89)   2.2s status=success
[ 10/13] ldbc_q12 -> cypher 

ok iters=1 tokens=(  130,  94)   2.2s status=success
[ 11/13] ldbc_q13 -> cypher 

ok iters=1 tokens=(  182, 173)   2.9s status=success
[ 12/13] ldbc_q14 -> cypher 

ok iters=1 tokens=(   89,  58)   2.1s status=success
[ 13/13] ldbc_q15 -> cypher 

ok iters=1 tokens=(  101,  78)   2.6s status=success
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_cypher_claude-opus-4-8.json


## Run-level summary

In [ ]:
import pandas as pd

df = pd.DataFrame(load_records(OUTPUTS_DIR))
if len(df):
    df['billed_input_tokens'] = df['input_tokens'] + df['cache_read_tokens'] + df['cache_creation_tokens']
    print(f'Total records: {len(df)}')
    print(f"Validation passed: {int(df['validation_passed'].sum())} / {len(df)}")
    print(f"Total tokens: {int(df['billed_input_tokens'].sum()):,} in / {int(df['output_tokens'].sum()):,} out")
    print(f"Mean duration: {df['duration_seconds'].mean():.2f}s")
    display(df.groupby(['dataset','target','model']).agg(
        n=('query_id','count'), passed=('validation_passed','sum'),
        mean_iter=('iterations_used','mean'),
        in_tok=('billed_input_tokens','sum'), out_tok=('output_tokens','sum')))
else:
    print('No records yet.')

Records written. Proceed to `02_behavioural_metrics.ipynb`.